# PART A

 imports

In [84]:
from sklearn.datasets import load_diabetes,load_breast_cancer,load_digits,load_iris
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score,accuracy_score,confusion_matrix as sklearn_confusion_matrix,precision_score,recall_score,f1_score

# 1

In [85]:
diabetes=load_diabetes(scaled=False)
x_diabetes=diabetes.data
y_diabetes=diabetes.target

breast_cancer=load_breast_cancer()
x_cancer=breast_cancer.data
y_cancer=1-breast_cancer.target

digits=load_digits()
x_digits=digits.data/16
y_digits=digits.target

iris=load_iris()
x_iris=iris.data
y_iris=iris.target


## split function

In [86]:
def split_data(X, y, regression=False):

    stratify = None if regression else y
    
    X_train_pool, X_test, y_train_pool, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=stratify
    )
    
    stratify_train = None if regression else y_train_pool

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_pool, y_train_pool,
        test_size=0.25,
        random_state=42,
        stratify=stratify_train
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

## Split datasets

In [87]:
datasets = {
    "Diabetes": (x_diabetes, y_diabetes, True),
    "Breast Cancer": (x_cancer, y_cancer, False),
    "Digits": (x_digits, y_digits, False),
    "Iris": (x_iris, y_iris, False)
}

splits = {}

for name, (X, y, regression) in datasets.items():

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, regression)

    splits[name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test
    }

In [88]:
for name, data in splits.items():

    print(f"\n{name}")
    
    for split in ["train", "val", "test"]:

        y = data[f"y_{split}"]

        print(
            f"{split}: size={len(y)}, "
            f"class counts={np.bincount(y) if name != 'Diabetes' else 'N/A'}"
        )


Diabetes
train: size=264, class counts=N/A
val: size=89, class counts=N/A
test: size=89, class counts=N/A

Breast Cancer
train: size=341, class counts=[214 127]
val: size=114, class counts=[71 43]
test: size=114, class counts=[72 42]

Digits
train: size=1077, class counts=[106 109 107 109 109 109 109 107 104 108]
val: size=360, class counts=[36 37 35 37 36 36 36 36 35 36]
test: size=360, class counts=[36 36 35 37 36 37 36 36 35 36]

Iris
train: size=90, class counts=[30 30 30]
val: size=30, class counts=[10 10 10]
test: size=30, class counts=[10 10 10]


# 2 : implement standizer and add bias

In [89]:
class Standardizer:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1

        return self

    def transform(self, X):
        return (X - self.mean) / self.std
    

In [90]:
def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.concatenate((ones, X), axis=1)

## standarizing all datasets

In [91]:
scaled_splits = {}

for name, data in splits.items():

    X_train = data["X_train"]
    X_val = data["X_val"]
    X_test = data["X_test"]

    y_train = data["y_train"]
    y_val = data["y_val"]
    y_test = data["y_test"]

    # Digits are already divided by 16
    if name == "Digits":
        scaled_splits[name] = data.copy()
        continue

    scaler = Standardizer()

    X_train_scaled = scaler.fit(X_train).transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    scaled_splits[name] = {
        "X_train": X_train_scaled,
        "X_val": X_val_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test
    }

In [92]:
for name, data in scaled_splits.items():

    if name == "Digits":
        continue

    X_train_scaled = data["X_train"]

    means = np.mean(X_train_scaled, axis=0)
    stds = np.std(X_train_scaled, axis=0)

    original_stds = np.std(splits[name]["X_train"], axis=0)
    non_constant = original_stds != 0

    print(f"\n{name}")
    print("Maximum absolute mean:", np.max(np.abs(means)))
    print("Maximum standard deviation difference from 1:",
          np.max(np.abs(stds[non_constant] - 1)))

    assert np.all(np.abs(means) < 1e-9)
    assert np.allclose(stds[non_constant], 1)


Diabetes
Maximum absolute mean: 2.2372676102294822e-15
Maximum standard deviation difference from 1: 2.1094237467877974e-15

Breast Cancer
Maximum absolute mean: 5.108979384873301e-15
Maximum standard deviation difference from 1: 8.881784197001252e-16

Iris
Maximum absolute mean: 1.2927930331190711e-15
Maximum standard deviation difference from 1: 4.440892098500626e-16


In [93]:
for name, data in splits.items():

    X_train = data["X_train"]

    X_bias = add_bias(X_train)

    print(f"\n{name}")
    print("Original shape:", X_train.shape)
    print("Shape after adding bias:", X_bias.shape)


Diabetes
Original shape: (264, 10)
Shape after adding bias: (264, 11)

Breast Cancer
Original shape: (341, 30)
Shape after adding bias: (341, 31)

Digits
Original shape: (1077, 64)
Shape after adding bias: (1077, 65)

Iris
Original shape: (90, 4)
Shape after adding bias: (90, 5)


The mean and standard deviation must be calculated using training data only to prevent data leakage. Using validation or test statistics introduces information from unseen data into preprocessing, which can make the final test evaluation misleading.

# 3

In [ ]:
def mse(y, pred):
    return np.mean((y - pred) ** 2)

def rmse(y, pred):
    return np.sqrt(mse(y, pred))

def r2(y, pred):

    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - ss_res / ss_tot

def accuracy(y, pred):
    return np.mean(y == pred)

def confusion_matrix(y, pred, K):

    cm = np.bincount(K * y + pred,minlength=K * K).reshape(K, K)

    return cm

def classification_metrics(cm):

    K = cm.shape[0]

    precision = np.zeros(K)
    recall = np.zeros(K)
    f1 = np.zeros(K)

    for i in range(K):

        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP

        if TP + FP == 0:
            precision[i] = 0
        else:
            precision[i] = TP / (TP + FP)

        if TP + FN == 0:
            recall[i] = 0
        else:
            recall[i] = TP / (TP + FN)

        if precision[i] + recall[i] == 0:
            f1[i] = 0
        else:
            f1[i] = 2 * precision[i] * recall[i] / (
                precision[i] + recall[i]
            )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_precision": np.mean(precision),
        "macro_recall": np.mean(recall),
        "macro_f1": np.mean(f1)
    }


    

In [95]:
rng = np.random.default_rng(42)
N = 500
K = 4
y = rng.integers(0, K, N)
pred = rng.integers(0, K, N)


## verifying classification metrics

In [96]:
cm = confusion_matrix(y, pred, K)

cm_sk = sklearn_confusion_matrix(
    y, pred, labels=np.arange(K)
)

metrics = classification_metrics(cm)

assert np.isclose(
    accuracy(y, pred),
    accuracy_score(y, pred),
    atol=1e-12
)

assert np.allclose(cm, cm_sk, atol=1e-12)

assert np.allclose(
    metrics["precision"],
    precision_score(y, pred, average=None, zero_division=0),
    atol=1e-12
)

assert np.allclose(
    metrics["recall"],
    recall_score(y, pred, average=None, zero_division=0),
    atol=1e-12
)

assert np.allclose(
    metrics["f1"],
    f1_score(y, pred, average=None, zero_division=0),
    atol=1e-12
)

assert np.isclose(
    metrics["macro_precision"],
    precision_score(y, pred, average="macro", zero_division=0),
    atol=1e-12
)

assert np.isclose(
    metrics["macro_recall"],
    recall_score(y, pred, average="macro", zero_division=0),
    atol=1e-12
)

assert np.isclose(
    metrics["macro_f1"],
    f1_score(y, pred, average="macro", zero_division=0),
    atol=1e-12
)

print("All classification metrics passed!")

All classification metrics passed!


## verifying regression metrics

In [97]:
y_reg = rng.normal(size=N)
pred_reg = rng.normal(size=N)

assert np.isclose(
    mse(y_reg, pred_reg),
    mean_squared_error(y_reg, pred_reg),
    atol=1e-12
)

assert np.isclose(
    rmse(y_reg, pred_reg),
    np.sqrt(mean_squared_error(y_reg, pred_reg)),
    atol=1e-12
)

assert np.isclose(
    r2(y_reg, pred_reg),
    r2_score(y_reg, pred_reg),
    atol=1e-12
)

print("All regression metrics passed!")

All regression metrics passed!


# 4

In [98]:
def numerical_gradient(f, w, epsilon=1e-5):

    grad = np.zeros_like(w, dtype=float)

    for i in np.ndindex(w.shape):

        w_plus = w.copy()
        w_minus = w.copy()

        w_plus[i] += epsilon
        w_minus[i] -= epsilon

        grad[i] = (
            f(w_plus) - f(w_minus)
        ) / (2 * epsilon)

    return grad

In [99]:
def grad_check(f, grad, w, epsilon=1e-5):

    analytical = grad(w)

    numerical = numerical_gradient(f, w, epsilon)

    numerator = np.linalg.norm(analytical - numerical)

    denominator = max(
        np.linalg.norm(analytical),
        np.linalg.norm(numerical),
        1e-12
    )

    return numerator / denominator

In [100]:
def test_function(w):

    return w[0]**2 + 3*w[1]**2 + 2*w[2]**2


def test_gradient(w):

    return np.array([
        2*w[0],
        6*w[1],
        4*w[2]
    ])


rng = np.random.default_rng(42)

w = rng.normal(0, 0.1, size=3)

error = grad_check(
    test_function,
    test_gradient,
    w
)

print("Relative error:", error)

assert error < 1e-6

print("Gradient check passed!")

Relative error: 7.32301050694436e-13
Gradient check passed!
